In [2]:
# aomori_lawson_to_excel.py
import re
import time
from urllib.parse import urljoin, urlparse, urlunparse

import httpx
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://www.navitime.co.jp"
PREF_URL = f"{BASE}/category/0201001009/02/"  # 青森県のローソン一覧（県ページ）

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/120.0.0.0 Safari/537.36"),
    "Referer": "https://www.navitime.co.jp/"
}
SLEEP = 1.0
TIMEOUT = 30

# ユーザー指定（短い表記）
REQUESTED_SHORT = [
    "青森市","弘前市","八戸市","黒石市","五所川原市","十和田市","三沢市","むつ市",
    "つがる市","平川市","平内町","鰺ヶ沢町","西目屋村","藤崎町","鶴田町",
    "六戸町","東通村","三戸町",
]

# NAVITIMEの郡名つき表記へマップ（都市はそのまま）
SHORT_TO_LONG = {
    "平内町": "東津軽郡平内町",
    "鰺ヶ沢町": "西津軽郡鰺ヶ沢町",
    "西目屋村": "中津軽郡西目屋村",
    "藤崎町": "南津軽郡藤崎町",
    "鶴田町": "北津軽郡鶴田町",
    "六戸町": "上北郡六戸町",
    "東通村": "下北郡東通村",
    "三戸町": "三戸郡三戸町",
}
TARGETS_LONG = []
for name in REQUESTED_SHORT:
    TARGETS_LONG.append(SHORT_TO_LONG.get(name, name))
TARGETS_LONG = list(dict.fromkeys(TARGETS_LONG))  # 重複削除

def strip_query(u: str) -> str:
    """URLのクエリ・フラグメントを除去（?page=2 を殺す）"""
    p = urlparse(u)
    return urlunparse((p.scheme, p.netloc, p.path, "", "", ""))

def get(url: str, *, tries: int = 3):
    """httpx GET with retry"""
    last = None
    for i in range(1, tries + 1):
        try:
            r = httpx.get(url, headers=HEADERS, timeout=TIMEOUT)
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            print(f"[RETRY {i}/{tries}] {url} -> {e}")
            time.sleep(1.2 * i)
    raise last

def soup_of(url: str) -> BeautifulSoup:
    html = get(url).text
    return BeautifulSoup(html, "html.parser")

CITY_URL_RE = re.compile(r"/category/0201001009/02\d{3}/$")

def collect_city_links():
    """県ページから市町村リンク（5桁コード終端のみ）を収集（郡名つき名に正規化して返す）"""
    soup = soup_of(PREF_URL)
    links = {}
    found_names = set()
    for a in soup.select('a[href*="/category/0201001009/"]'):
        name = a.get_text(strip=True)
        href = a.get("href") or ""
        # 絶対URL + クエリ除去
        href = strip_query(urljoin(BASE, href))
        # /02xxx/ で終わるURLだけ採用（ページネーション等を除外）
        if not CITY_URL_RE.search(href):
            continue

        # そのまま郡名つきで一致
        if name in TARGETS_LONG:
            links[name] = href
            found_names.add(name)
            continue

        # 「郡名つき」⇄「短い表記」を吸収（例：北津軽郡鶴田町 ←→ 鶴田町）
        for short, long_ in SHORT_TO_LONG.items():
            if name.endswith(short) and long_ in TARGETS_LONG:
                links[long_] = href
                found_names.add(long_)
                break

        # 都市（郡名なし）の厳密一致
        for t in TARGETS_LONG:
            if t == name:
                links[t] = href
                found_names.add(t)
                break

    missing = [t for t in TARGETS_LONG if t not in found_names]
    print(f"[INFO] 市町村リンク取得: {len(links)}件  / 未検出: {missing}")
    return links

def iter_list_pages(list_url: str):
    """?page=N のページを順に巡回。list_url はクエリ無しが前提。"""
    base = strip_query(list_url)  # 念のため再除去（安全網）
    page = 1
    while True:
        url = base if page == 1 else f"{base}?page={page}"
        r = get(url)
        s = BeautifulSoup(r.text, "html.parser")

        # 店舗詳細へのリンク候補を抽出
        cards = s.select("li[class*='SpotListItem'], article[class*='SpotListItem'], li, article")
        detail_links = []
        for c in cards:
            a = c.select_one('a[href*="/poi?spot="]')
            title = c.select_one("h3, h2, .SpotListItem_title, a")
            if a and title and "ローソン" in title.get_text(strip=True):
                detail_links.append(urljoin(BASE, a.get("href")))
        detail_links = list(dict.fromkeys(detail_links))

        if not detail_links:
            break
        yield detail_links
        page += 1
        time.sleep(SLEEP)

def parse_detail(detail_url: str):
    """詳細ページから 店名(h1) と 住所 を取得"""
    r = get(detail_url)
    s = BeautifulSoup(r.text, "html.parser")

    # 店名
    h1 = s.find("h1")
    name = re.sub(r"\s+", " ", h1.get_text(" ", strip=True)) if h1 else None

    # 住所（dt「住所」→ dd、もしくは「住所」ラベルの次要素、フォールバックで「青森県…」）
    addr = None
    dt = s.find("dt", string=re.compile(r"^\s*住所\s*$"))
    if dt:
        dd = dt.find_next_sibling("dd")
        if dd:
            addr = dd.get_text(strip=True)
    if not addr:
        lab = s.find(string=re.compile(r"^\s*住所\s*$"))
        if lab:
            nxt = lab.find_next()
            if nxt:
                addr = nxt.get_text(strip=True)
    if not addr:
        m = s.find(string=re.compile(r"^青森県"))
        if m:
            addr = m.strip()

    return name, addr

def main(out_xlsx="lawson_aomori.xlsx"):
    city_links = collect_city_links()
    if not city_links:
        print("[WARN] 市町村リンクが0件です。県ページDOM変更や表記ゆれを疑ってください。")
        return

    rows = []
    for area, list_url in city_links.items():
        print(f"[AREA] {area} -> {list_url}")
        total_area = 0
        for detail_links in iter_list_pages(list_url):
            print(f"  [PAGE] 店舗候補: {len(detail_links)}件")
            for d in detail_links:
                try:
                    name, addr = parse_detail(d)
                except Exception as e:
                    print(f"    [ERR] {d} {e}")
                    name, addr = None, None
                if name and addr:
                    rows.append({"エリア": area, "店舗名": name, "住所": addr, "詳細URL": d})
                    total_area += 1
                time.sleep(SLEEP)
        print(f"  -> 収集 {total_area}件")
        time.sleep(SLEEP)

    df = pd.DataFrame(rows)
    if df.empty:
        print("[WARN] 結果が空です。セレクタ不一致・ブロック・該当0件の可能性があります。")
    df.to_excel(out_xlsx, index=False)
    print(f"[DONE] 保存: {out_xlsx}  件数: {len(df)}")

if __name__ == "__main__":
    main()

[INFO] 市町村リンク取得: 0件  / 未検出: ['青森市', '弘前市', '八戸市', '黒石市', '五所川原市', '十和田市', '三沢市', 'むつ市', 'つがる市', '平川市', '東津軽郡平内町', '西津軽郡鰺ヶ沢町', '中津軽郡西目屋村', '南津軽郡藤崎町', '北津軽郡鶴田町', '上北郡六戸町', '下北郡東通村', '三戸郡三戸町']
[WARN] 市町村リンクが0件です。県ページDOM変更や表記ゆれを疑ってください。


In [3]:
import re
import time
from urllib.parse import urljoin, urlparse, urlunparse

import httpx
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://www.navitime.co.jp"
PREF_URL = f"{BASE}/category/0201001009/02/"  # 青森県のローソン一覧（県ページ）
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/120.0.0.0 Safari/537.36"),
    "Referer": "https://www.navitime.co.jp/"
}
TIMEOUT = 30
SLEEP = 1.0

# ユーザー指定の18エリア（住所に含まれる文字列として使う）
TARGETS_ADDR_KEYWORDS = [
    "青森市","弘前市","八戸市","黒石市","五所川原市","十和田市","三沢市","むつ市",
    "つがる市","平川市",
    "東津軽郡平内町","西津軽郡鰺ヶ沢町","中津軽郡西目屋村",
    "南津軽郡藤崎町","北津軽郡鶴田町",
    "上北郡六戸町","下北郡東通村","三戸郡三戸町",
]

def strip_query(u: str) -> str:
    p = urlparse(u)
    return urlunparse((p.scheme, p.netloc, p.path, "", "", ""))

def get(url: str, *, tries: int = 3):
    last = None
    for i in range(1, tries+1):
        try:
            r = httpx.get(url, headers=HEADERS, timeout=TIMEOUT)
            r.raise_for_status()
            return r
        except Exception as e:
            last = e
            print(f"[RETRY {i}/{tries}] {url} -> {e}")
            time.sleep(1.2 * i)
    raise last

def iter_pref_pages(pref_url: str):
    """県ページを?page=Nで順に。カードが無くなったところで終了。"""
    base = strip_query(pref_url)
    page = 1
    while True:
        url = base if page == 1 else f"{base}?page={page}"
        r = get(url)
        soup = BeautifulSoup(r.text, "html.parser")

        # 店舗カードから詳細リンク候補
        cards = soup.select("li[class*='SpotListItem'], article[class*='SpotListItem'], li, article")
        detail_links = []
        for c in cards:
            a = c.select_one('a[href*="/poi?spot="]')
            t = c.select_one("h3, h2, .SpotListItem_title, a")
            if a and t and "ローソン" in t.get_text(strip=True):
                detail_links.append(urljoin(BASE, a.get("href")))
        detail_links = list(dict.fromkeys(detail_links))

        if not detail_links:
            break

        print(f"[PAGE] {page} -> 店舗候補 {len(detail_links)}件")
        yield detail_links
        page += 1
        time.sleep(SLEEP)

def parse_detail(detail_url: str):
    """詳細ページから 店名(h1) と 住所 を取得"""
    r = get(detail_url)
    s = BeautifulSoup(r.text, "html.parser")
    # 店名
    h1 = s.find("h1")
    name = re.sub(r"\s+", " ", h1.get_text(" ", strip=True)) if h1 else None
    # 住所
    addr = None
    dt = s.find("dt", string=re.compile(r"^\s*住所\s*$"))
    if dt:
        dd = dt.find_next_sibling("dd")
        if dd:
            addr = dd.get_text(strip=True)
    if not addr:
        lab = s.find(string=re.compile(r"^\s*住所\s*$"))
        if lab:
            nxt = lab.find_next()
            if nxt:
                addr = nxt.get_text(strip=True)
    if not addr:
        m = s.find(string=re.compile(r"^青森県"))
        if m:
            addr = m.strip()
    return name, addr

def area_from_address(addr: str) -> str | None:
    """住所に含まれるキーワードからエリア名を返す（最初にヒットしたもの）"""
    for kw in TARGETS_ADDR_KEYWORDS:
        if kw in addr:
            return kw
    return None

def main(out_xlsx="lawson_aomori.xlsx"):
    rows = []
    total = 0
    for detail_links in iter_pref_pages(PREF_URL):
        for d in detail_links:
            try:
                name, addr = parse_detail(d)
            except Exception as e:
                print(f"  [ERR] {d} {e}")
                name, addr = None, None
            if name and addr:
                area = area_from_address(addr)
                # 対象エリアに該当するものだけ採用
                if area is not None:
                    rows.append({"エリア": area, "店舗名": name, "住所": addr, "詳細URL": d})
                    total += 1
            time.sleep(SLEEP)

    df = pd.DataFrame(rows)
    if df.empty:
        print("[WARN] 該当エリアの結果が0件でした。キーワードやセレクタを確認してください。")
    df.to_excel(out_xlsx, index=False)
    print(f"[DONE] 保存: {out_xlsx}  件数: {len(df)}（総候補から住所フィルタ後）")

if __name__ == "__main__":
    main()

[PAGE] 1 -> 店舗候補 15件
[PAGE] 2 -> 店舗候補 15件
[PAGE] 3 -> 店舗候補 15件
[PAGE] 4 -> 店舗候補 15件
[PAGE] 5 -> 店舗候補 15件
[PAGE] 6 -> 店舗候補 15件
[PAGE] 7 -> 店舗候補 15件
[PAGE] 8 -> 店舗候補 15件
[PAGE] 9 -> 店舗候補 15件
[PAGE] 10 -> 店舗候補 15件
[PAGE] 11 -> 店舗候補 15件
[PAGE] 12 -> 店舗候補 15件
[PAGE] 13 -> 店舗候補 15件
[PAGE] 14 -> 店舗候補 15件
[PAGE] 15 -> 店舗候補 15件
[PAGE] 16 -> 店舗候補 15件
[PAGE] 17 -> 店舗候補 15件
[PAGE] 18 -> 店舗候補 15件
[PAGE] 19 -> 店舗候補 8件
[RETRY 1/3] https://www.navitime.co.jp/category/0201001009/02/?page=20 -> Client error '404 Not Found' for url 'https://www.navitime.co.jp/category/0201001009/02/?page=20'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
[RETRY 2/3] https://www.navitime.co.jp/category/0201001009/02/?page=20 -> Client error '404 Not Found' for url 'https://www.navitime.co.jp/category/0201001009/02/?page=20'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404
[RETRY 3/3] https://www.navitime.co.jp/category/0201001009/02/?page=20

HTTPStatusError: Client error '404 Not Found' for url 'https://www.navitime.co.jp/category/0201001009/02/?page=20'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

In [4]:
# aomori_lawson_to_excel_final.py
import re
import time
from urllib.parse import urljoin, urlparse, urlunparse

import httpx
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://www.navitime.co.jp"
PREF_URL = f"{BASE}/category/0201001009/02/"  # 青森県のローソン一覧（県ページ）

HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/120.0.0.0 Safari/537.36"),
    "Referer": "https://www.navitime.co.jp/"
}
TIMEOUT = 30
SLEEP = 1.0
MAX_PAGES = 50  # 念のため上限

TARGETS_ADDR_KEYWORDS = [
    "青森市","弘前市","八戸市","黒石市","五所川原市","十和田市","三沢市","むつ市",
    "つがる市","平川市",
    "東津軽郡平内町","西津軽郡鰺ヶ沢町","中津軽郡西目屋村",
    "南津軽郡藤崎町","北津軽郡鶴田町",
    "上北郡六戸町","下北郡東通村","三戸郡三戸町",
]

class EndOfPages(Exception):
    """ページ終端（404など）を示す内部用例外"""

def strip_query(u: str) -> str:
    p = urlparse(u)
    return urlunparse((p.scheme, p.netloc, p.path, "", "", ""))

def get(url: str):
    """404 を終端として扱い、それ以外は通常どおりエラー"""
    r = httpx.get(url, headers=HEADERS, timeout=TIMEOUT)
    if r.status_code == 404:
        # ここが今回の肝：最終ページの次で 404 が返る → 終了シグナルにする
        raise EndOfPages(f"404 end: {url}")
    r.raise_for_status()
    return r

def iter_pref_pages(pref_url: str):
    """県ページを?page=Nで順巡。404 が来たら終了。"""
    base = strip_query(pref_url)
    for page in range(1, MAX_PAGES + 1):
        url = base if page == 1 else f"{base}?page={page}"
        try:
            r = get(url)
        except EndOfPages:
            # 最終ページの次に到達 → 正常終了
            print(f"[END] reached 404 at page {page}")
            break
        soup = BeautifulSoup(r.text, "html.parser")

        # 店舗カードから詳細リンク候補
        cards = soup.select("li[class*='SpotListItem'], article[class*='SpotListItem'], li, article")
        detail_links = []
        for c in cards:
            a = c.select_one('a[href*="/poi?spot="]')
            t = c.select_one("h3, h2, .SpotListItem_title, a")
            if a and t and "ローソン" in t.get_text(strip=True):
                detail_links.append(urljoin(BASE, a.get("href")))
        detail_links = list(dict.fromkeys(detail_links))

        if not detail_links:
            # （サイトの挙動によっては 404 ではなく 0 件になる場合もある）
            print(f"[END] page {page} has 0 candidates")
            break

        print(f"[PAGE] {page} -> 店舗候補 {len(detail_links)}件")
        yield detail_links
        time.sleep(SLEEP)

def parse_detail(detail_url: str):
    """詳細ページから 店名(h1) と 住所 を取得"""
    r = get(detail_url)
    s = BeautifulSoup(r.text, "html.parser")
    # 店名
    h1 = s.find("h1")
    name = re.sub(r"\s+", " ", h1.get_text(" ", strip=True)) if h1 else None
    # 住所
    addr = None
    dt = s.find("dt", string=re.compile(r"^\s*住所\s*$"))
    if dt:
        dd = dt.find_next_sibling("dd")
        if dd:
            addr = dd.get_text(strip=True)
    if not addr:
        lab = s.find(string=re.compile(r"^\s*住所\s*$"))
        if lab:
            nxt = lab.find_next()
            if nxt:
                addr = nxt.get_text(strip=True)
    if not addr:
        m = s.find(string=re.compile(r"^青森県"))
        if m:
            addr = m.strip()
    return name, addr

def area_from_address(addr: str) -> str | None:
    for kw in TARGETS_ADDR_KEYWORDS:
        if kw in addr:
            return kw
    return None

def main(out_xlsx="lawson_aomori.xlsx"):
    rows = []
    total = 0
    for detail_links in iter_pref_pages(PREF_URL):
        for d in detail_links:
            try:
                name, addr = parse_detail(d)
            except EndOfPages:
                # 詳細側でまれに 404 になっても終端扱いでスキップ
                continue
            except Exception as e:
                print(f"  [ERR] {d} {e}")
                continue
            if name and addr:
                area = area_from_address(addr)
                if area is not None:
                    rows.append({"エリア": area, "店舗名": name, "住所": addr, "詳細URL": d})
                    total += 1
        time.sleep(SLEEP)

    df = pd.DataFrame(rows)
    if df.empty:
        print("[WARN] 該当エリアの結果が0件でした。キーワードやセレクタを確認してください。")
    df.to_excel(out_xlsx, index=False)
    print(f"[DONE] 保存: {out_xlsx}  件数: {len(df)}")

if __name__ == "__main__":
    main()


[PAGE] 1 -> 店舗候補 15件
[PAGE] 2 -> 店舗候補 15件
[PAGE] 3 -> 店舗候補 15件
[PAGE] 4 -> 店舗候補 15件
[PAGE] 5 -> 店舗候補 15件
[PAGE] 6 -> 店舗候補 15件
[PAGE] 7 -> 店舗候補 15件
[PAGE] 8 -> 店舗候補 15件
[PAGE] 9 -> 店舗候補 15件
[PAGE] 10 -> 店舗候補 15件
[PAGE] 11 -> 店舗候補 15件
[PAGE] 12 -> 店舗候補 15件
[PAGE] 13 -> 店舗候補 15件
[PAGE] 14 -> 店舗候補 15件
[PAGE] 15 -> 店舗候補 15件
[PAGE] 16 -> 店舗候補 15件
[PAGE] 17 -> 店舗候補 15件
[PAGE] 18 -> 店舗候補 15件
[PAGE] 19 -> 店舗候補 8件
[END] reached 404 at page 20
[DONE] 保存: lawson_aomori.xlsx  件数: 239


In [5]:
前提として間違いがありました、

SyntaxError: invalid character '、' (U+3001) (2623827138.py, line 1)

In [6]:
# aomori_lawson_one_per_area.py
import re
import time
from urllib.parse import urljoin, urlparse, urlunparse

import httpx
import pandas as pd
from bs4 import BeautifulSoup

BASE = "https://www.navitime.co.jp"
PREF_URL = f"{BASE}/category/0201001009/02/"  # 青森県×ローソンの県ページ
HEADERS = {
    "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                   "AppleWebKit/537.36 (KHTML, like Gecko) "
                   "Chrome/120.0.0.0 Safari/537.36"),
    "Referer": "https://www.navitime.co.jp/"
}
TIMEOUT = 30
SLEEP = 1.0
MAX_PAGES = 50  # 念のため上限

# 各エリアにつき1件だけ取得（この順でExcelに並べます）
TARGETS = [
    "青森市","弘前市","八戸市","黒石市","五所川原市","十和田市","三沢市","むつ市",
    "つがる市","平川市",
    "東津軽郡平内町","西津軽郡鰺ヶ沢町","中津軽郡西目屋村",
    "南津軽郡藤崎町","北津軽郡鶴田町",
    "上北郡六戸町","下北郡東通村","三戸郡三戸町",
]

class EndOfPages(Exception):
    pass

def strip_query(u: str) -> str:
    p = urlparse(u)
    return urlunparse((p.scheme, p.netloc, p.path, "", "", ""))

def get(url: str):
    r = httpx.get(url, headers=HEADERS, timeout=TIMEOUT)
    if r.status_code == 404:
        raise EndOfPages(f"404 end: {url}")
    r.raise_for_status()
    return r

def iter_pref_pages(pref_url: str):
    base = strip_query(pref_url)
    for page in range(1, MAX_PAGES + 1):
        url = base if page == 1 else f"{base}?page={page}"
        try:
            r = get(url)
        except EndOfPages:
            print(f"[END] reached 404 at page {page}")
            break
        soup = BeautifulSoup(r.text, "html.parser")
        cards = soup.select("li[class*='SpotListItem'], article[class*='SpotListItem'], li, article")

        detail_links = []
        for c in cards:
            a = c.select_one('a[href*="/poi?spot="]')
            t = c.select_one("h3, h2, .SpotListItem_title, a")
            if a and t and "ローソン" in t.get_text(strip=True):
                detail_links.append(urljoin(BASE, a.get("href")))
        detail_links = list(dict.fromkeys(detail_links))

        if not detail_links:
            print(f"[END] page {page} has 0 candidates")
            break

        print(f"[PAGE] {page} -> 店舗候補 {len(detail_links)}件")
        yield detail_links
        time.sleep(SLEEP)

def parse_detail(detail_url: str):
    r = get(detail_url)
    s = BeautifulSoup(r.text, "html.parser")
    h1 = s.find("h1")
    name = re.sub(r"\s+", " ", h1.get_text(" ", strip=True)) if h1 else None

    addr = None
    dt = s.find("dt", string=re.compile(r"^\s*住所\s*$"))
    if dt:
        dd = dt.find_next_sibling("dd")
        if dd:
            addr = dd.get_text(strip=True)
    if not addr:
        lab = s.find(string=re.compile(r"^\s*住所\s*$"))
        if lab:
            nxt = lab.find_next()
            if nxt:
                addr = nxt.get_text(strip=True)
    if not addr:
        m = s.find(string=re.compile(r"^青森県"))
        if m:
            addr = m.strip()
    return name, addr

def area_from_address(addr: str) -> str | None:
    # 住所文字列から最初にヒットしたエリア名を返す
    for kw in TARGETS:
        if kw in addr:
            return kw
    return None

def main(out_xlsx="lawson_aomori_one_per_area.xlsx"):
    # まだ埋まっていないエリア集合
    remaining = set(TARGETS)
    # area → 1件だけのレコードを保存
    picked = {}

    for detail_links in iter_pref_pages(PREF_URL):
        for d in detail_links:
            # 全部埋まったら即終了
            if not remaining:
                break
            try:
                name, addr = parse_detail(d)
            except EndOfPages:
                continue
            except Exception as e:
                print(f"  [ERR] {d} {e}")
                continue

            if not (name and addr):
                continue

            area = area_from_address(addr)
            if area and area in remaining:
                picked[area] = {
                    "エリア": area,
                    "店舗名": name,
                    "住所": addr,
                    "詳細URL": d
                }
                remaining.remove(area)
                print(f"  [PICK] {area}: {name}")

        if not remaining:
            print("[DONE] すべてのエリアで1件ずつ取得しました。")
            break

    # 見つからなかったエリアは「該当なし」で埋める
    rows = []
    for area in TARGETS:
        if area in picked:
            rows.append(picked[area])
        else:
            rows.append({
                "エリア": area,
                "店舗名": "該当なし",
                "住所": "",
                "詳細URL": ""
            })

    df = pd.DataFrame(rows)
    df.to_excel(out_xlsx, index=False)
    print(f"[SAVE] {out_xlsx}  件数: {len(df)}  未取得: {sorted(list(remaining))}")

if __name__ == "__main__":
    main()

[PAGE] 1 -> 店舗候補 15件
  [PICK] 青森市: ローソン 青森篠田店
  [PICK] 八戸市: ローソン 八戸西郵便局前店
  [PICK] 弘前市: ローソン 弘前茂森町店
  [PICK] 北津軽郡鶴田町: ローソン 鶴田町店
[PAGE] 2 -> 店舗候補 15件
  [PICK] 五所川原市: ローソン 金木町店
  [PICK] 三沢市: ローソン 三沢美野原店
  [PICK] 平川市: ローソン 平賀町店
  [PICK] 上北郡六戸町: ローソン 六戸町店
  [PICK] 東津軽郡平内町: ローソン 平内藤沢店
[PAGE] 3 -> 店舗候補 15件
  [PICK] むつ市: ローソン むつ中央店
  [PICK] つがる市: ローソン 木造町店
  [PICK] 南津軽郡藤崎町: ローソン 常盤榊店
[PAGE] 4 -> 店舗候補 15件
  [PICK] 十和田市: ローソン 十和田三小通店
[PAGE] 5 -> 店舗候補 15件
[PAGE] 6 -> 店舗候補 15件
  [PICK] 黒石市: ローソン 黒石浅瀬石店
[PAGE] 7 -> 店舗候補 15件
[PAGE] 8 -> 店舗候補 15件
[PAGE] 9 -> 店舗候補 15件
[PAGE] 10 -> 店舗候補 15件
[PAGE] 11 -> 店舗候補 15件
[PAGE] 12 -> 店舗候補 15件
[PAGE] 13 -> 店舗候補 15件
  [PICK] 西津軽郡鰺ヶ沢町: ローソン 鰺ヶ沢駅前店
[PAGE] 14 -> 店舗候補 15件
[PAGE] 15 -> 店舗候補 15件
[PAGE] 16 -> 店舗候補 15件
[PAGE] 17 -> 店舗候補 15件
  [PICK] 三戸郡三戸町: ローソン 三戸二日町店
[PAGE] 18 -> 店舗候補 15件
[PAGE] 19 -> 店舗候補 8件
[END] reached 404 at page 20
[SAVE] lawson_aomori_one_per_area.xlsx  件数: 18  未取得: ['下北郡東通村', '中津軽郡西目屋村']
